In [0]:
# REPORTE DE EJECUCION — resumen del pipeline completo
# genera el JSON final donde se documentan los resultados


import json
from datetime import datetime
from pyspark.sql import functions as f

CATALOGO     = "nyc_taxi_andres"
CAPA_RAW     = f"{CATALOGO}.raw"
CAPA_TRUSTED = f"{CATALOGO}.trusted"
CAPA_REFINED = f"{CATALOGO}.refined"

# leemos los datos que ya tenemos en cada capa
viajes_raw     = spark.table(f"{CAPA_RAW}.viajes_enero_2023")
viajes_trusted = spark.table(f"{CAPA_TRUSTED}.viajes_limpios")
kpi_demanda    = spark.table(f"{CAPA_REFINED}.kpi_demanda_temporal")
kpi_zonas      = spark.table(f"{CAPA_REFINED}.kpi_top10_zonas_rentables")
kpi_calidad    = spark.table(f"{CAPA_REFINED}.kpi_calidad_datos")

# calculamos algunos numeros para el reporte
total_raw       = viajes_raw.count()
total_trusted   = viajes_trusted.count()
total_descartados = total_raw - total_trusted
pct_descartado  = round((total_descartados / total_raw) * 100, 2)

ingresos_limpios = viajes_trusted.agg(
    f.round(f.sum("total_cobrado"), 2).alias("total")
).collect()[0]["total"]

# zona mas rentable
zona_top = kpi_zonas.orderBy(f.desc("ingreso_promedio")).first()

# franja con mas viajes
franja_top = kpi_demanda.orderBy(f.desc("cantidad_viajes")).first()

# armamos el reporte
reporte = {
    "pipeline"   : "NYC Yellow Taxi — Arquitectura Medallion",
    "fecha_reporte": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "fuente"     : "TLC Trip Record Data — Enero 2023",
    "capas": {
        "raw": {
            "tabla_viajes" : f"{CAPA_RAW}.viajes_enero_2023",
            "tabla_zonas"  : f"{CAPA_RAW}.zonas_taxi",
            "total_viajes" : total_raw,
            "total_zonas"  : 265
        },
        "trusted": {
            "tabla"              : f"{CAPA_TRUSTED}.viajes_limpios",
            "viajes_entrada"     : total_raw,
            "viajes_descartados" : total_descartados,
            "pct_descartado"     : pct_descartado,
            "viajes_validos"     : total_trusted,
            "detalle_descartados": {
                "tiempo_invalido"     : 1121,
                "sin_distancia"       : 44814,
                "tarifa_invalida"     : 22522,
                "outliers_extremos"   : 3046,
                "nulos_columnas_clave": 0
            }
        },
        "refined": {
            "tablas_generadas": [
                "kpi_demanda_temporal",
                "kpi_top10_zonas_rentables",
                "kpi_calidad_datos",
                "reporte_calidad_datos"
            ],
            "kpi_demanda_temporal": {
                "franja_con_mas_viajes": franja_top["franja_horaria"],
                "dia_con_mas_viajes"   : franja_top["dia_semana"],
                "cantidad_viajes_pico" : franja_top["cantidad_viajes"]
            },
            "kpi_zonas_rentables": {
                "zona_top"            : zona_top["zona_origen"],
                "barrio_top"          : zona_top["barrio_origen"],
                "ingreso_promedio_top": float(zona_top["ingreso_promedio"])
            },
            "kpi_calidad_datos": {
                "ingresos_datos_limpios"    : float(ingresos_limpios),
                "ingresos_datos_crudos"     : 82865192.22,
                "diferencia_ingresos"       : round(82865192.22 - float(ingresos_limpios), 2),
                "pct_impacto_en_ingresos"   : 1.2
            }
        }
    },
    "gobierno": {
        "catalogo"    : CATALOGO,
        "schemas"     : ["raw", "trusted", "refined"],
        "cdes_documentados"   : 3,
        "terminos_glosario"   : 5,
        "reglas_calidad"      : 4
    }
}

# ya con esto se muestra el reporte en pantalla mas organizado
print(json.dumps(reporte, indent=4, ensure_ascii=False))

In [0]:
# guardamos el JSON en el volumen de Unity Catalog
# desde ahi lo copiamos manualmente al repo en GitHub

ruta_json = "/Volumes/nyc_taxi_andres/raw/archivos_fuente/reporte_ejecucion.json"

with open(ruta_json, "w", encoding="utf-8") as f:
    json.dump(reporte, f, indent=4, ensure_ascii=False)

print(f"Reporte guardado en: {ruta_json}")
